# Async Central Management Console Demo

This notebook demonstrates how to use the async versions of the Fabric API functions for better performance when making multiple API calls.

## Setup and Imports

In [ ]:
import asyncio
import aiohttp
import pandas as pd
import time
from datetime import datetime

# Import your async functions (assuming they're in the same directory or properly imported)
# from notebook_content import *

## Configuration Parameters

In [ ]:
# Configuration parameters
kv_uri = 'https://kvfabricprodeus2rh.vault.azure.net/'
client_id_secret = 'fuam-spn-client-id'
tenant_id_secret = 'fuam-spn-tenant-id'
client_secret_name = 'fuam-spn-secret'

workspace_id = 'a046cf0f-8dca-4b61-b95e-7adf68fb4b0a'
dataset_id = '708da792-a344-4079-b205-61c587a51600'

## Example 1: Basic Async Usage

In [ ]:
async def basic_async_example():
    """Basic example of using async functions"""
    print(f"Starting basic async example at {datetime.now()}")
    
    # Get authentication token
    token = await get_api_token_via_akv_async(kv_uri, client_id_secret, tenant_id_secret, client_secret_name)
    print("✓ Authentication token obtained")
    
    # Create HTTP session with timeout and connection pooling
    timeout = aiohttp.ClientTimeout(total=300, connect=60)
    connector = aiohttp.TCPConnector(limit=10, limit_per_host=5)
    
    async with aiohttp.ClientSession(timeout=timeout, connector=connector) as session:
        # Get dataset refresh information
        refresh_info = await get_dataset_refresh_info_async(session, workspace_id, dataset_id, token)
        print(f"✓ Retrieved refresh info: {len(refresh_info)} records")
        
        # Get all workspaces
        workspaces = await get_all_workspaces_async(session, token)
        workspace_count = len(workspaces.get('workspaces', [])) if workspaces else 0
        print(f"✓ Retrieved workspaces: {workspace_count} workspaces")
    
    print(f"Completed basic async example at {datetime.now()}")
    return refresh_info, workspaces

# Run the basic example
refresh_data, workspace_data = await basic_async_example()

## Example 2: Concurrent Operations - Performance Comparison

In [ ]:
async def performance_comparison_example():
    """Compare sync vs async performance for multiple API calls"""
    print("\n=== Performance Comparison: Sync vs Async ===")
    
    # Get authentication token
    token = await get_api_token_via_akv_async(kv_uri, client_id_secret, tenant_id_secret, client_secret_name)
    
    # Define multiple operations to perform
    operations = [
        ('get_workspaces', lambda session: get_all_workspaces_async(session, token)),
        ('get_connections', lambda session: get_all_connections_async(session, token)),
        ('get_refresh_info', lambda session: get_dataset_refresh_info_async(session, workspace_id, dataset_id, token))
    ]
    
    # Method 1: Sequential (simulating sync behavior)
    print("\n🔄 Running operations sequentially...")
    start_time = time.time()
    
    timeout = aiohttp.ClientTimeout(total=300, connect=60)
    connector = aiohttp.TCPConnector(limit=10, limit_per_host=5)
    
    sequential_results = {}
    async with aiohttp.ClientSession(timeout=timeout, connector=connector) as session:
        for op_name, op_func in operations:
            try:
                result = await op_func(session)
                sequential_results[op_name] = result
                print(f"  ✓ {op_name} completed")
            except Exception as e:
                print(f"  ✗ {op_name} failed: {str(e)}")
                sequential_results[op_name] = None
    
    sequential_time = time.time() - start_time
    print(f"Sequential execution time: {sequential_time:.2f} seconds")
    
    # Method 2: Concurrent (true async)
    print("\n⚡ Running operations concurrently...")
    start_time = time.time()
    
    async with aiohttp.ClientSession(timeout=timeout, connector=connector) as session:
        # Create all tasks
        tasks = {op_name: op_func(session) for op_name, op_func in operations}
        
        # Execute all tasks concurrently
        concurrent_results = {}
        for op_name, task in tasks.items():
            try:
                result = await task
                concurrent_results[op_name] = result
                print(f"  ✓ {op_name} completed")
            except Exception as e:
                print(f"  ✗ {op_name} failed: {str(e)}")
                concurrent_results[op_name] = None
    
    concurrent_time = time.time() - start_time
    print(f"Concurrent execution time: {concurrent_time:.2f} seconds")
    
    # Performance improvement
    improvement = ((sequential_time - concurrent_time) / sequential_time) * 100
    print(f"\n📊 Performance improvement: {improvement:.1f}% faster with async")
    
    return sequential_results, concurrent_results

# Run the performance comparison
seq_results, con_results = await performance_comparison_example()

## Example 3: Bulk Dataset Operations

In [ ]:
async def bulk_dataset_operations_example():
    """Example of processing multiple datasets concurrently"""
    print("\n=== Bulk Dataset Operations ===")
    
    # Get authentication token
    token = await get_api_token_via_akv_async(kv_uri, client_id_secret, tenant_id_secret, client_secret_name)
    
    # Define multiple workspace-dataset pairs to process
    # In a real scenario, you might get these from the workspaces API
    workspace_dataset_pairs = [
        (workspace_id, dataset_id),
        # Add more pairs as needed
        # ('another-workspace-id', 'another-dataset-id'),
    ]
    
    print(f"Processing {len(workspace_dataset_pairs)} datasets...")
    
    # Use the bulk operation function
    start_time = time.time()
    bulk_results = await get_multiple_datasets_refresh_info_async(workspace_dataset_pairs, token)
    execution_time = time.time() - start_time
    
    print(f"\n📊 Bulk operation completed in {execution_time:.2f} seconds")
    
    # Display results summary
    for key, df in bulk_results.items():
        if not df.empty:
            print(f"  ✓ {key}: {len(df)} refresh records")
        else:
            print(f"  ⚠ {key}: No data or error occurred")
    
    return bulk_results

# Run the bulk operations example
bulk_data = await bulk_dataset_operations_example()

## Example 4: Error Handling and Resilience

In [ ]:
async def error_handling_example():
    """Demonstrate robust error handling in async operations"""
    print("\n=== Error Handling and Resilience ===")
    
    # Get authentication token
    token = await get_api_token_via_akv_async(kv_uri, client_id_secret, tenant_id_secret, client_secret_name)
    
    # Mix of valid and invalid operations to test error handling
    test_operations = [
        ('valid_workspace', workspace_id),
        ('invalid_workspace', 'invalid-workspace-id-12345'),
    ]
    
    timeout = aiohttp.ClientTimeout(total=300, connect=60)
    connector = aiohttp.TCPConnector(limit=10, limit_per_host=5)
    
    async with aiohttp.ClientSession(timeout=timeout, connector=connector) as session:
        for op_name, ws_id in test_operations:
            print(f"\nTesting {op_name}...")
            try:
                result = await get_all_datasets_in_workspace_async(session, ws_id, token)
                if result:
                    dataset_count = len(result.get('value', []))
                    print(f"  ✓ Success: Found {dataset_count} datasets")
                else:
                    print(f"  ⚠ No data returned")
            except Exception as e:
                print(f"  ✗ Error handled gracefully: {type(e).__name__}: {str(e)}")
    
    print("\n✓ Error handling demonstration completed")

# Run the error handling example
await error_handling_example()

## Example 5: Complete Workflow with All Async Operations

In [ ]:
async def complete_workflow_example():
    """Complete workflow using all async functions"""
    print("\n=== Complete Async Workflow ===")
    
    # Step 1: Authentication
    print("🔐 Step 1: Getting authentication token...")
    token = await get_api_token_via_akv_async(kv_uri, client_id_secret, tenant_id_secret, client_secret_name)
    print("  ✓ Authentication successful")
    
    # Step 2: Bulk workspace operations
    print("\n🏢 Step 2: Getting workspace and connection information...")
    bulk_results = await bulk_workspace_operations_async(token)
    
    workspaces = bulk_results.get('workspaces')
    connections = bulk_results.get('connections')
    
    if workspaces:
        workspace_count = len(workspaces.get('workspaces', []))
        print(f"  ✓ Retrieved {workspace_count} workspaces")
    
    if connections:
        connection_count = len(connections.get('value', []))
        print(f"  ✓ Retrieved {connection_count} connections")
    
    # Step 3: Dataset operations
    print("\n📊 Step 3: Processing dataset information...")
    
    timeout = aiohttp.ClientTimeout(total=300, connect=60)
    connector = aiohttp.TCPConnector(limit=10, limit_per_host=5)
    
    async with aiohttp.ClientSession(timeout=timeout, connector=connector) as session:
        # Get datasets in workspace
        datasets = await get_all_datasets_in_workspace_async(session, workspace_id, token)
        if datasets:
            dataset_count = len(datasets.get('value', []))
            print(f"  ✓ Found {dataset_count} datasets in workspace")
        
        # Get refresh information for our target dataset
        refresh_info = await get_dataset_refresh_info_async(session, workspace_id, dataset_id, token)
        print(f"  ✓ Retrieved refresh history: {len(refresh_info)} records")
        
        # Optional: Start a refresh (uncomment if needed)
        # refresh_result = await start_dataset_refresh_async(session, workspace_id, dataset_id, token)
        # print(f"  ✓ Refresh operation: {refresh_result['status']}")
    
    print("\n🎉 Complete workflow finished successfully!")
    
    return {
        'workspaces': workspaces,
        'connections': connections,
        'datasets': datasets,
        'refresh_info': refresh_info
    }

# Run the complete workflow
workflow_results = await complete_workflow_example()

## Summary and Best Practices

### Key Benefits of Async Implementation:

1. **Performance**: Concurrent execution can significantly reduce total execution time
2. **Resource Efficiency**: Better utilization of I/O wait time
3. **Scalability**: Handle multiple API calls more efficiently
4. **Resilience**: Better error handling and timeout management

### Best Practices Implemented:

1. **Connection Pooling**: Using `aiohttp.TCPConnector` with limits
2. **Timeout Management**: Configured timeouts for both connection and total request time
3. **Error Handling**: Comprehensive exception handling with logging
4. **Resource Management**: Proper session management with async context managers
5. **Batch Operations**: Functions to handle multiple operations concurrently

### When to Use Async vs Sync:

- **Use Async**: Multiple API calls, batch operations, real-time applications
- **Use Sync**: Simple single operations, quick scripts, debugging

Both versions are available in the codebase for flexibility!